# Week 8: Networks II: Transit

The other networked structures we deal with often in urban data science are transit maps. Like roads, each transit line can be represented by a `LineString`, and each stop is a point along it. When transit stops have multiple lines running through it, it can become a network. 

Transit data is often stored for most cities in a format called General Transit Feed Specfication (GTFS). This format encodes the stops, lines, route schedules, and agency information in one place. 

For today, I have pre-downloaded the GTFS for Boston, on Canvas under Data as `MBTA_GTFS.zip`. Save this into the same folder as this notebook.

## 1 Getting to GTFS static data

Often times we want the transit schedules and stop locations as they are printed on the map in each city. This is called **static** data. It doesn't change day to day unless the agency produces an update. 

GTFS feeds for most cities are provided at https://mobilitydatabase.org/ or https://www.transit.land/feeds or often at the agency's website itself (for example this one for Boston: https://www.mbta.com/developers/gtfs). 

They are saved as .zip files containing the following:
* agency.txt — required, transit agencies with service represented in this dataset
* stops.txt — required, stops where vehicles pick up or drop off riders. Also defines stations and station entrances
* routes.txt — required, transit routes. A route is a group of trips that are displayed to riders as a single service
* trips.txt — required, trips for each route. A trip is a sequence of two or more stops that occur during a specific time period
* stop_times.txt — required, times that a vehicle arrives at and departs from stops for each trip
* calendar.txt — conditionally required, service dates specified using a weekly schedule with start and end dates
* calendar_dates.txt — conditionally required, exceptions for the services defined in the calendar.txt
* feed_info.txt — optional, dataset metadata, including publisher, version, and expiration information

Here's an [example](https://gtfs.org/getting-started/example-feed/) of what the inside of the .txt files look like. 

## 2 Reading in GTFS

There are two ways I recommend reading in the GTFS zip files:
1. Unzip the folder and read in each .txt file to pandas
2. Use a prebuilt package called `gtfs_kit`

Let's walk through the benefits and cons of each.

### 2.1 Reading in .txt files


In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# There are a few main files we might care about
# Each stop as a row
stops = pd.read_csv('MBTA_GTFS/stops.txt')
stops.head()

In [ ]:
# Each named route in a transit system
routes = pd.read_csv('MBTA_GTFS/routes.txt')
routes.head()

In [ ]:
# The geometry of each route
shapes = pd.read_csv('MBTA_GTFS/shapes.txt')
shapes.head()

In [ ]:
# Each trip of a route
trips = pd.read_csv('MBTA_GTFS/trips.txt')
trips.head()

In [ ]:
# For each trip, the information about when the transit vehicle comes and goes to a stop
stop_times = pd.read_csv('MBTA_GTFS/stop_times.txt')
stop_times.head()

So to recap, here are some important connector columns between the files: 
* `route_id`: connects `routes` and `trips`
* `trip_id`: connects `trips` and `stop_times`
* `shape_id`: connects `shapes` and `trips`
* `stop_id`: connects `stops` and `stop_times`

### 2.2 Installing gtfs_kit

`gtfs_kit` is a repbuilt package that has some functionality built in to it

In [ ]:
!pip install gtfs_kit

In [ ]:
import gtfs_kit as gkt

In [ ]:
# Now we read in directly from the zip file and all of the files are taken in
feed = gkt.read_feed("MBTA_GTFS.zip", dist_units="ft")

In [ ]:
# You can get basic descriptive stats about the network 
feed.describe()

In [ ]:
# All of the same files are read in
feed.stops.head()

In [ ]:
feed.routes.head()

In [ ]:
feed.shapes.head()

In [ ]:
feed.trips.head()

In [ ]:
feed.stop_times.head()

In [ ]:
## YOUR TURN 
## Determine how many unique trips are taken on the "Red" route in Boston



## 3 Connecting between the files

What if you want to know the stops of a transit line? Or maybe just the subway systems? You need to get from routes to stops via the intermediate paths. 

Recall, here are some important connector columns between the files: 
* `route_id`: connects `routes` and `trips`
* `trip_id`: connects `trips` and `stop_times`
* `shape_id`: connects `shapes` and `trips`
* `stop_id`: connects `stops` and `stop_times`


So, to get geometry values for the routes, we need to follow the path:

`routes:route_id` $\rightarrow$ `trips:route_id` 

`trips:trip_id` $\rightarrow$ `stop_times:trip_id` 

`stop_times:stop_id` $\rightarrow$ `stops:stop_id`

In [ ]:
# Let's look at routes again
routes.head()

In [ ]:
# What types of transit are there? 
routes.route_desc.value_counts()

In [ ]:
# First, let's keep only the rapid transit lines
major_routes = routes[routes.route_desc == 'Rapid Transit']

# Then let's connect it to trips via route_id
major_trips = trips[trips['route_id'].isin(major_routes['route_id'].unique())]
print(len(major_trips))
major_trips.head()

In [ ]:
# Now let's connect trips to stop_times via trip_id
major_stop_times = stop_times[stop_times['trip_id'].isin(major_trips['trip_id'].values)]
len(major_stop_times)
major_stop_times.head()

In [ ]:
# Finally, let's connect stop_times to stops via stop_id
major_stops = stops[stops['stop_id'].isin(stop_times['stop_id'].values)]
len(major_stops)
major_stops.head()

In [ ]:
## YOUR TURN 
## Turn major_stops into a geodataframe and plot the location of the stops


# Network Review and Practice

Let's put both of our pieces together now. 

Your goal is to locate the nearest rapid transit station in Boston, Massachusetts from the following locations, then map out the shortest (by distance) walking path to get there using `osmnx`. 

Think about the steps you will need to go through:
* Identify the rapid transit stations
* Find the minimum distance between our location of interest and the stations
* Pull the walking network for Boston
* Get the closest nodes to the origin (location) and destination (station)
* Plot the shortest path

Location 1: Grocery store at lon: `-71.011028`, lat: `42.427241`

Location 2: Beach at lon: `-71.016152`, lat: `42.280145`